# Step 04 — Few-Shot Prompting

🇬🇧 **English** (this notebook) · 🇩🇪 Deutsch — not yet available.

Provide the model with 2–3 examples of input→output pairs before asking your real question. The model learns the expected format and style from the examples — no training, no fine-tuning, just context.

## Learning objective

By the end of this notebook, you will:

- Understand what "few-shot prompting" (in-context learning) means and how it differs from zero-shot
- Have built a message list that mixes example turns with a real question
- Be able to judge whether examples actually steer a model's output, or just add noise

## Prerequisites

- [Step 03 — Zero-Shot Prompting](step_03_zero_shot_prompting.ipynb) completed, with your team's topic already chosen — this notebook reuses it
- The same `.env` setup as Step 03 (a working `GEMINI_API_KEY` or `OPENAI_API_KEY`)

## Background

Few-shot prompting shows the model 2–3 example input→output pairs before your real question, so it learns the expected format and style from context alone — no training, no weight updates. This is called **in-context learning**: the examples live inside the prompt, not inside the model.

**Few-shot prompting** was the central finding of:

> Brown, T., et al. (2020). *Language Models are Few-Shot Learners*. NeurIPS 2020. [arXiv:2005.14165](https://arxiv.org/abs/2005.14165)

A prompt built this way is really a small program — the examples are the instructions, and the model is the runtime executing them.

## How this works

Run the Setup cell once, then the exercise cell:

1. **Setup** loads your API keys and creates the `llm` object — identical to Step 03, just without the display code, since it's reused below.
2. **The exercise cell** builds a `messages` list with *real* `user` → `assistant` turns for each example, followed by one final `user` turn holding your actual question. Each example pair is one "shot" — two examples here means this is two-shot prompting.
3. The model sees the whole list as one conversation and continues it — so it naturally follows the pattern the example turns established.

In [1]:
import os

# CrewAI's telemetry tries to reach its backend over the network on import;
# on a restricted/firewalled connection this can hang for a long time with no
# error. Disable it before crewai is imported.
os.environ.setdefault('CREWAI_DISABLE_TELEMETRY', 'true')

from dotenv import load_dotenv
from crewai import LLM
from IPython.display import Markdown, display

load_dotenv()

llm = LLM(model=os.getenv("MODEL", "gemini/gemini-3.1-flash-lite"))

Each example is a real `user` → `assistant` turn, not text glued into one message: the "user" turn poses the example input, and the "assistant" turn shows the ideal output, exactly as if the model had already answered it. The final `user` turn is the real question, left for the model to complete.

In [2]:
# ── Examples — show the model what good output looks like ────────────────────
# Each pair is one "shot", encoded as a real user->assistant turn.
# Two examples = two-shot prompting. Notice the format they teach: one-sentence
# definition, then a simple analogy - a different, simpler topic than the real question.
example_1_input  = "Explain what a firewall is."
example_1_output = (
    "A firewall is a security barrier that monitors and controls network traffic "
    "based on preset rules. Think of it like a security guard at a building's "
    "entrance, checking IDs and only letting authorized visitors through."
)

example_2_input  = "Explain what encryption is."
example_2_output = (
    "Encryption is the process of converting readable data into a coded format "
    "that only authorized parties can decode. Think of it like writing a message "
    "in a secret language that only the sender and intended recipient understand."
)

# ── The actual question — the final "user" turn, left for the model to complete ──
user_message = "Explain the EU AI Act in simple terms in a short paragraph."

# ── Roles — no persona needed, so "system" is left out; "assistant" is used
# for real this time, holding the ideal answer to each example "user" turn ────
output = llm.call(
    messages=[
        {"role": "user", "content": example_1_input},
        {"role": "assistant", "content": example_1_output},
        {"role": "user", "content": example_2_input},
        {"role": "assistant", "content": example_2_output},
        {"role": "user", "content": user_message},
    ]
)
display(Markdown(output))

The EU AI Act is the world’s first comprehensive law designed to regulate artificial intelligence by categorizing it based on risk. It bans AI systems that pose an "unacceptable" threat to safety or human rights, sets strict transparency and safety requirements for "high-risk" systems (like those used in healthcare or hiring), and requires minimal rules for lower-risk tools (like chatbots). Its main goal is to ensure that AI developed and used in Europe is safe, ethical, and respects fundamental human rights.

## Your task

1. Run the cells above, in order (Setup first, then the exercise cell).

2. Does the model's EU AI Act answer follow the same "definition + analogy" format the two examples taught it, even though neither example ever mentioned the EU AI Act?

3. Now change one example's format (e.g. drop the analogy from `example_2_output`, leaving just a definition) and re-run. Does the final answer follow the *new*, inconsistent pattern, or average the two?

4. Swap `user_message` for a question about your own team's topic — same topic as Step 03.

5. Fill in the **Step 04** section of `EVALUATION.md`.

## Shortcomings

Examples only steer the model as far as they're consistent and well-chosen — a sloppy or contradictory example teaches a sloppy or contradictory pattern (task 3 above just asked you to try exactly that). Writing good examples by hand also doesn't scale: for a complex task you'd want many examples, covering edge cases, which makes the prompt long and expensive, and you're still hand-authoring both the input *and* the ideal output yourself.

[Step 05](step_05_prompt_template.ipynb) takes a different, more structured approach to shaping output: instead of examples, it splits the prompt into explicit named components (persona, instructions, format).

## Resources for further reading

- Brown, T., et al. (2020). *Language Models are Few-Shot Learners*. NeurIPS 2020. [arXiv:2005.14165](https://arxiv.org/abs/2005.14165) — introduced few-shot / in-context learning as a distinct capability of large language models.
- Liu, P., et al. (2023). *Pre-train, Prompt, and Predict: A Systematic Survey of Prompting Methods in Natural Language Processing*. ACM Computing Surveys, 55(9), 1–35. [arXiv:2107.13586](https://arxiv.org/abs/2107.13586) — a broader survey of prompting strategies, of which few-shot is one.